## Imports

In [ ]:
import numpy as np
from matplotlib import pyplot as plt

import torch
from torch_geometric.data import Data
from scipy.spatial import Delaunay

import networkx as nx
from torch_geometric.utils import to_networkx
import matplotlib.patches as mpatches

## GRAPHS FROM SYNTHETIC DATA

In [ ]:
synthetic_1 = np.load('../data/preprocessing/normalized/normalized_local_norotation_close.npy')
synthetic_2 = np.load('../data/preprocessing/normalized/normalized_local_norotation_far.npy')
original = np.load('../data/preprocessing/normalized/normalized_local_norotation_original.npy')

In [ ]:
synthetic_1 = np.load('../data/preprocessing/graph/new_close.npy')
synthetic_2 = np.load('../data/preprocessing/graph/new_far.npy')
original = np.load('../data/preprocessing/graph/original.npy')

In [ ]:
def point_to_polyline_distance_vectorized(points, polyline):

    A = polyline[:-1]        # (M-1, 2)
    B = polyline[1:]         # (M-1, 2)
    AB = B - A               # (M-1, 2)
    AB_norm_sq = np.sum(AB ** 2, axis=1)  # (M-1,)

    # Expand dims for broadcasting
    P = points[:, None, :]   # (N,1,2)
    A = A[None, :, :]        # (1,M-1,2)
    AB = AB[None, :, :]      # (1,M-1,2)

    AP = P - A                      # (N,M-1,2)
    t = np.sum(AP * AB, axis=2) / (AB_norm_sq + 1e-12)  # (N, M-1)
    t = np.clip(t, 0.0, 1.0)

    proj = A + t[..., None] * AB    # (N,M-1,2)
    dist = np.linalg.norm(P - proj, axis=2)  # (N, M-1)

    return np.min(dist, axis=1)  # (N,)

In [ ]:
def perpendicular_distance(points, polyline):

    # Segment directions
    v = polyline[1:] - polyline[:-1]           # (N-1,2)
    v = np.vstack([v, v[-1]])                  # pad last

    # Normal vectors (rotate by 90°)
    normals = np.stack([-v[:,1], v[:,0]], axis=1)
    normals /= (np.linalg.norm(normals, axis=1, keepdims=True) + 1e-12)

    # Vector from polyline to points
    diff = points - polyline                   # (N,2)

    # Signed perpendicular distance
    dist = np.sum(diff * normals, axis=1)
    return dist


In [ ]:
def compute_orientation_angles(coords):
    
    dx = np.diff(coords[:, 0], append=coords[-1, 0])
    dy = np.diff(coords[:, 1], append=coords[-1, 1])
    angles = np.arctan2(dy, dx)
    angles = (angles + np.pi) / (2 * np.pi)  # normalize 0–1

    return angles.reshape(-1, 1)

In [ ]:
def compute_orientation_features(coords, return_curvature=False):

    # Forward differences
    deltas = np.diff(coords, axis=0)

    # Repeat last direction to keep same length
    deltas = np.vstack([deltas, deltas[-1]])

    # Normalize direction vectors
    norms = np.linalg.norm(deltas, axis=1, keepdims=True) + 1e-12
    tangents = deltas / norms

    # Angle (for curvature computation only)
    angles = np.arctan2(tangents[:, 1], tangents[:, 0])

    # Orientation features (recommended)
    orient = np.stack([np.cos(angles), np.sin(angles)], axis=1)

    if not return_curvature:
        return orient

    # Curvature = change in angle (wrapped)
    dtheta = np.diff(angles, prepend=angles[0])
    dtheta = (dtheta + np.pi) % (2 * np.pi) - np.pi

    curvature = dtheta.reshape(-1, 1)

    return orient, curvature


In [ ]:
def make_graphs(original, synthetic_1, synthetic_2, graph_type):
    
    num_seqs = original.shape[0]
    graphs = []

    for seq_idx in range(num_seqs):
        seq_len = original.shape[1]   # auto-detect length
         
        # Extract polylines
        orig = original[seq_idx][:seq_len]         # (N,2)
        syn1 = synthetic_1[seq_idx][:seq_len]      # (N,2)
        syn2 = synthetic_2[seq_idx][:seq_len]      # (N,2)
         
        # Compute node features
        feats = []

        # ----- Original line features -----
        angles_o, curvature_o = compute_orientation_features(orig, return_curvature=True)
        dist_o = perpendicular_distance(orig, syn1).reshape(-1, 1)

        feats_o = np.hstack([
            np.zeros((seq_len, 1)),      # line_id = 0
            orig,                        # x,y
            angles_o,                    # orientation
            curvature_o,                 # curvature 
            dist_o                       # dist to synthetic_1
        ])
        feats.append(feats_o)

        # ----- Synthetic_1 line features -----
        angles_s1, curvature_s1 = compute_orientation_features(syn1, return_curvature=True)
        dist_s1 = perpendicular_distance(syn1, orig).reshape(-1, 1)

        feats_s1 = np.hstack([
            np.ones((seq_len, 1)),       # line_id = 1
            syn1,
            angles_s1,
            curvature_s1,
            dist_s1
        ])
        feats.append(feats_s1)

        # Stack node features
        x = np.vstack(feats)  # (2*N, feature_dim)
         
        # Build edges
        edge_src = []
        edge_dst = []

        # Sequential edges in original line
        idx_offset_o = 0
        for i in range(seq_len - 1):
            edge_src += [idx_offset_o + i, idx_offset_o + i + 1]
            edge_dst += [idx_offset_o + i + 1, idx_offset_o + i]

       # Sequential edges in synthetic_1 line
        idx_offset_s1 = seq_len
        for i in range(seq_len - 1):
            edge_src += [idx_offset_s1 + i, idx_offset_s1 + i + 1]
            edge_dst += [idx_offset_s1 + i + 1, idx_offset_s1 + i]
        
        # Cross Edges original[i] <-> synthetic_1[i]
        if graph_type in ('sequential'):
            for i in range(seq_len):
                edge_src += [i, idx_offset_s1 + i]
                edge_dst += [idx_offset_s1 + i, i]
        
        # Delaunay edges
        if graph_type in ('delaunay', 'hybrid'):
            all_points = np.vstack([orig, syn1])  # shape: (2N,2)
            tri = Delaunay(all_points)
            triangles = tri.simplices  # (T,3)

            tri_edges = set()
            for a, b, c in triangles:
                tri_edges.add(tuple(sorted((a, b))))
                tri_edges.add(tuple(sorted((b, c))))
                tri_edges.add(tuple(sorted((c, a))))

            for u, v in tri_edges:
                edge_src += [u, v]
                edge_dst += [v, u]

        # Final edge index
        #edge_index = np.vstack([edge_src, edge_dst]).astype(np.int64)
        edges = set(zip(edge_src, edge_dst))
        edge_src, edge_dst = zip(*edges)
        edge_index = np.vstack([edge_src, edge_dst]).astype(np.int64)
         
        # Target: shift of synthetic_1 → synthetic_2
        shift = syn2 - syn1         # (N,2)

        y = np.zeros((2 * seq_len, 2), dtype=np.float32)
        y[idx_offset_s1:idx_offset_s1 + seq_len] = shift  # only syn1 nodes
         
        # Convert to PyTorch
        data = Data(
            x=torch.tensor(x, dtype=torch.float),
            edge_index=torch.tensor(edge_index, dtype=torch.long),
            y=torch.tensor(y, dtype=torch.float)
        )
        graphs.append(data)

    return graphs


In [ ]:
graphs_seq2 = make_graphs(original, synthetic_1, synthetic_2, 'sequential')

In [ ]:
graphs_delaunay2 = make_graphs(original, synthetic_1, synthetic_2, 'delaunay')

### Save Results

In [ ]:
#torch.save(graphs_seq, f'../data/final_dataset/graph/graphs_sequential_norotation_local_new.pt')
#torch.save(graphs_delaunay, f'../data/final_dataset/graph/graphs_delaunay_norotation_local_new.pt')
graphs_seq = torch.load('../data/final_dataset/graph/graphs_sequential_norotation_local_new.pt', weights_only=False)
graphs_delaunay = torch.load('../data/final_dataset/graph/graphs_delaunay_norotation_local_new.pt', weights_only=False)

### Ploting Graphs

In [ ]:
def plot_graph(data, index, saving_img=False):
    # Convert PyG Data -> NetworkX
    G = to_networkx(data, to_undirected=True)
    pos = {i: (float(data.x[i][1]), float(data.x[i][2])) for i in range(data.num_nodes)}

    # Node colors by line_id
    color_map = ["#143642", "#EC9A29"]
    colors = [color_map[int(data.x[i][0].item())] for i in range(data.num_nodes)]

    plt.figure(figsize=(8, 7))
    plt.title(f'Constructed Graph with Sequential Lines - Line {index}')
    plt.xlabel('X')
    plt.ylabel('Y')

    # Draw graph
    nx.draw(
        G, pos,
        node_size=5,
        node_color=colors,
        edge_color="#B3B5B6A6",
        alpha=0.8
    )

    # ---- Plot shift vectors as arrows ----
    xy = data.x[:, 1:3].cpu().numpy()
    dxy = data.y.cpu().numpy()

    plt.quiver(
        xy[:, 0], xy[:, 1],          # start points
        dxy[:, 0], dxy[:, 1],        # direction (dx, dy)
        angles='xy', scale_units='xy', scale=1,
        width=0.002, color='#0F8B8D', alpha=0.3
    )

    # ---- Legend ----
    legend_patches = [
        mpatches.Patch(color=color_map[0], label='Line A'),
        mpatches.Patch(color=color_map[1], label='Line B')
    ]
    plt.legend(handles=legend_patches, title="Line IDs")

    plt.axis('equal')
    
    if saving_img:
        save_path = f'../data/figures/graph_sequential.png'
        plt.savefig(save_path, dpi=300, bbox_inches='tight')

    plt.show()


In [ ]:
graph = graphs_delaunay2

for i in range(102,123):
    plot_graph(graphs_seq2[i], i, )
    plot_graph(graphs_delaunay2[i], i)


In [ ]:
torch.save(graphs_seq2, f'../data/final_dataset/graph/graphs_sequential2.pt')
torch.save(graphs_delaunay2, f'../data/final_dataset/graph/graphs_delaunay2.pt')

## REALWORLD DATA

In [ ]:
from scipy.interpolate import interp1d

def resample_line(coords, n_points=64):
    """
    coords: np.array (N,2)
    Returns: np.array (n_points,2) equally spaced along line length
    """
    # Compute cumulative distance along the line
    dist = np.sqrt(np.sum(np.diff(coords, axis=0)**2, axis=1))
    dist = np.insert(dist, 0, 0)
    cumdist = np.cumsum(dist)
    cumdist /= cumdist[-1]  # normalize 0..1

    # Interpolate x and y
    f_x = interp1d(cumdist, coords[:,0])
    f_y = interp1d(cumdist, coords[:,1])
    new_dist = np.linspace(0,1,n_points)
    new_coords = np.vstack([f_x(new_dist), f_y(new_dist)]).T
    return new_coords

In [ ]:
import geopandas as gpd
import numpy as np
import torch
from torch_geometric.data import Data
from scipy.spatial import Delaunay

# -------------------------------
# 0. Settings / flags
# -------------------------------
shp_path = "/Users/pia/Documents/Cartography/Masterarbeit/GIS_Project/push/selected_2.shp"
line_column = 'line_type'
n_points = 128

use_sequential = True
use_cross = True
use_delaunay = False

# -------------------------------
# 1. Load shapefile
# -------------------------------
gdf = gpd.read_file(shp_path)

# -------------------------------
# 2. Extract coordinates by line
# -------------------------------
coords_by_line = {}
for idx, row in gdf.iterrows():
    line_id = row[line_column]
    x, y = row['geometry'].xy
    coords_by_line[line_id] = np.array([x, y]).T  # shape (N,2)

# Example resample function, replace with your own
def resample_line(points, n_points):
    """Resample points along the line to n_points"""
    idx = np.linspace(0, len(points)-1, n_points)
    x_new = np.interp(idx, np.arange(len(points)), points[:,0])
    y_new = np.interp(idx, np.arange(len(points)), points[:,1])
    return np.vstack([x_new, y_new]).T

orig = resample_line(coords_by_line[0], n_points)
syn1 = resample_line(coords_by_line[1], n_points)
syn2 = resample_line(coords_by_line[2], n_points)

# -------------------------------
# 3. Normalize coordinates to [0,1]
# -------------------------------
all_coords = np.vstack([orig, syn1, syn2])
coords_min = all_coords.min(axis=0)
coords_max = all_coords.max(axis=0)

orig = (orig - coords_min) / (coords_max - coords_min + 1e-8)
syn1 = (syn1 - coords_min) / (coords_max - coords_min + 1e-8)
syn2 = (syn2 - coords_min) / (coords_max - coords_min + 1e-8)

seq_len = n_points  # all lines have same number of points

# -------------------------------
# 4. Compute node features
# -------------------------------

# Line 0 features
angles_o, curvature_o = compute_orientation_features(orig, return_curvature=True)
dist_o = perpendicular_distance(orig, syn1).reshape(-1,1)
feats_o = np.hstack([
    np.zeros((seq_len,1)),  # line_id = 0
    orig,
    angles_o,
    curvature_o,
    dist_o,
    np.zeros((seq_len, 1))
])

# Line 1 features
angles_s1, curvature_s1 = compute_orientation_features(syn1, return_curvature=True)
dist_s1 = perpendicular_distance(syn1, orig).reshape(-1,1)
feats_s1 = np.hstack([
    np.ones((seq_len,1)),   # line_id = 1
    syn1,
    angles_s1,
    curvature_s1,
    dist_s1,
    np.zeros((seq_len, 1))
])

# Combine node features
x = np.vstack([feats_o, feats_s1])

# -------------------------------
# 5. Build edges
# -------------------------------
def build_sequential_edges(seq_len, start_index=0):
    edges = []
    for i in range(seq_len - 1):
        edges += [ (start_index + i, start_index + i + 1), (start_index + i + 1, start_index + i) ]
    return edges

def build_cross_edges(seq_len, start_index1=0, start_index2=None):
    if start_index2 is None:
        start_index2 = seq_len
    edges = []
    for i in range(seq_len):
        edges += [ (start_index1 + i, start_index2 + i), (start_index2 + i, start_index1 + i) ]
    return edges

def delaunay_edges(points, start_index=0):
    tri = Delaunay(points)
    edges = set()
    for simplex in tri.simplices:
        for i in range(3):
            a = start_index + simplex[i]
            b = start_index + simplex[(i+1)%3]
            edges.add((a, b))
            edges.add((b, a))
    return edges

edges = set()

if use_sequential:
    edges.update(build_sequential_edges(seq_len, start_index=0))        # line 0
    edges.update(build_sequential_edges(seq_len, start_index=seq_len))  # line 1

if use_cross:
    edges.update(build_cross_edges(seq_len, start_index1=0, start_index2=seq_len))

if use_delaunay:
    edges.update(delaunay_edges(orig, start_index=0))
    edges.update(delaunay_edges(syn1, start_index=seq_len))

edge_src, edge_dst = zip(*edges)
edge_index = np.vstack([edge_src, edge_dst]).astype(np.int64)

# -------------------------------
# 6. Target y (shift of syn1 -> syn2)
# -------------------------------
y = np.zeros((2*seq_len, 2), dtype=np.float32)
y[seq_len:seq_len*2] = syn2 - syn1  # only line 1 nodes

# -------------------------------
# 7. Convert to PyTorch Geometric Data
# -------------------------------
data = Data(
    x=torch.tensor(x, dtype=torch.float),
    edge_index=torch.tensor(edge_index, dtype=torch.long),
    y=torch.tensor(y, dtype=torch.float)
)

print(data)


In [ ]:
plot_graph(data, 0)

In [ ]:
torch.save(data, f'../data/final_dataset/graph/push_data_22_seq.pt')